# H2. Evaluation as an experiment
Book: Model Selection, Overfitting and Underfitting, Cross-Validation.

Optional background: Alisa's math notes, [Bias & variance](https://alisawuffles.notion.site/math-notes#3737eb8736058073b5c2c112ba2cba13). Use the course book for split design and leakage.
These selected readings are optional support. The classroom examples define the required scope.

The regression test sample stays unused until the H3 choice is fixed.

## Setup
Run this cell once. Helpers support the experiments below.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({'figure.figsize': (8, 4.5), 'font.size': 12})

def regression_data(seed=600, n=100):
    rng = np.random.default_rng(seed)
    x = rng.uniform(-2, 2, n)
    y = 1 + 2*x + 0.7*x*x + rng.normal(0, 1, n)
    return x.reshape(-1, 1), y

def regression_split():
    x, y = regression_data()
    order = np.random.default_rng(17).permutation(len(y))
    a, b = order[:70], order[70:]
    return x[a], x[b], y[a], y[b]

def polynomial_model(degree=3, alpha=0):
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import PolynomialFeatures, StandardScaler
    from sklearn.linear_model import LinearRegression, Ridge
    estimator = LinearRegression() if alpha == 0 else Ridge(alpha=alpha)
    return make_pipeline(PolynomialFeatures(degree, include_bias=False),
                         StandardScaler(), estimator)

## A. Training performance and new observations (30 minutes)
Predict which polynomial degree will fit training observations best.
Do you expect the same ordering on validation data?

In [ ]:
from sklearn.metrics import mean_squared_error
xr, xv, yr, yv = regression_split()
degrees = [1, 3, 12]
fig, ax = plt.subplots()
grid = np.linspace(-2, 2, 300).reshape(-1, 1)
ax.scatter(xr[:, 0], yr, color="gray", alpha=.5, label="Training observations")
for degree in degrees:
    model = polynomial_model(degree).fit(xr, yr)
    train_mse = mean_squared_error(yr, model.predict(xr))
    val_mse = mean_squared_error(yv, model.predict(xv))
    print(degree, "train MSE", round(train_mse, 3),
          "validation MSE", round(val_mse, 3))
    ax.plot(grid[:, 0], model.predict(grid), label=f"Degree {degree}")
baseline = np.repeat(yr.mean(), len(yv))
print("Constant baseline validation MSE:", mean_squared_error(yv, baseline))
ax.set(xlabel="x", ylabel="Prediction", ylim=(-5, 12))
ax.legend()
plt.show()

Keep the same observations and metric across the comparison.
Explain what the data support and what a single split cannot establish.

Prediction:

Observation:

Explanation:

## B. The suspiciously good feature (30 minutes)
A colleague proposes a feature generated after the outcome is known.
Predict its validation error. Is a random split sufficient protection?

In [ ]:
from sklearn.linear_model import LinearRegression
rng = np.random.default_rng(603)
leaked_train = np.c_[xr, yr + rng.normal(0, .05, len(yr))]
leaked_val = np.c_[xv, yv + rng.normal(0, .05, len(yv))]
leaky = LinearRegression().fit(leaked_train, yr)
honest = LinearRegression().fit(xr, yr)
print("With outcome-derived feature:", mean_squared_error(yv, leaky.predict(leaked_val)))
print("Available-at-prediction-time feature:", mean_squared_error(yv, honest.predict(xv)))

Remove the outcome-derived feature and rerun. State the prediction time.
Give a plausible real example of this failure, such as a transaction status
that only appears after the event being predicted.

Prediction:

Observation:

Explanation:

## Optional: the same fitting pipeline inside each fold

In [ ]:
from sklearn.model_selection import KFold, cross_val_score
folds = KFold(5, shuffle=True, random_state=600)
scores = -cross_val_score(polynomial_model(3), xr, yr, cv=folds,
                         scoring="neg_mean_squared_error")
print("Fold MSE:", scores, "Mean:", scores.mean())

## Individual exit
A team chooses the best of 20 models using test error. What role has that
dataset actually played? Design a split for predicting next year's outcomes.